# Prepare and process histology data shards

This notebook runs the `histology_data` CLI in a Colab session. It builds a catalog, streams source images into bounded raw shards, verifies the release, then processes shards one at a time using `/content` for temporary work and Drive for committed outputs. `Metadata.xlsx` stays authoritative; ZIP archives supply image files only.

Build `histology-data-runtime.zip` with `python scripts/build_data_runtime.py`, then copy it, the metadata file, and source directory or ZIP archives to the Drive locations below. Keep every original source available until preparation is complete. A first pass must read the source archive; it does not extract the whole ZIP. Rerunning verifies committed work and resumes at the next shard.

Start with the tiny local smoke config before running full preparation on Colab. These outputs contain QC and lossless patches only. They do not contain encoder features or training labels, and shard boundaries are not patient splits.

In [ ]:
import importlib.util
import os
import stat
import subprocess
import sys
import zipfile
from pathlib import Path

if importlib.util.find_spec('google.colab') is None:
    raise RuntimeError('Run this full-data notebook in Google Colab. Use the local smoke CLI for development.')

from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/histology_data_shards')
RUNTIME_ZIP = DRIVE_ROOT / 'histology-data-runtime.zip'
RUNTIME_ROOT = Path('/content/histology-data-runtime')
METADATA = DRIVE_ROOT / 'input' / 'Metadata.xlsx'
SOURCE_DIRECTORY = DRIVE_ROOT / 'input' / 'images'  # Set to None when using source ZIPs.
SOURCE_ZIP_DIRECTORY = DRIVE_ROOT / 'input' / 'zips'
RAW_RELEASE = DRIVE_ROOT / 'raw-release'
PROCESSED_OUTPUT = DRIVE_ROOT / 'processed'
WORK_ROOT = Path('/content/histology-work')  # Temporary SSD staging and tile workspace.
KEEP_WORK = False  # Set True to retain verified staged source files on SSD after processing.

if not RUNTIME_ZIP.is_file():
    raise FileNotFoundError(f'Copy the runtime bundle to {RUNTIME_ZIP}')
if not METADATA.is_file():
    raise FileNotFoundError(f'Metadata file not found: {METADATA}')

WORK_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(RUNTIME_ZIP) as runtime_archive:
    for member in runtime_archive.infolist():
        member_path = Path(member.filename)
        member_mode = member.external_attr >> 16
        if (member_path.is_absolute() or '..' in member_path.parts or ':' in member.filename
                or '\\' in member.filename or stat.S_ISLNK(member_mode)):
            raise ValueError(f'Unsafe runtime archive member: {member.filename!r}')
    runtime_archive.extractall(RUNTIME_ROOT)

requirements = RUNTIME_ROOT / 'requirements-data.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(requirements)], check=True)
CLI_ENV = os.environ.copy()
old_pythonpath = CLI_ENV.get('PYTHONPATH', '')
CLI_ENV['PYTHONPATH'] = str(RUNTIME_ROOT) + (os.pathsep + old_pythonpath if old_pythonpath else '')

# SOURCE_SELECTION_ADAPTER_BEGIN
def select_image_sources(source_directory, source_zip_directory):
    selected = []
    if source_directory is not None and Path(source_directory).is_dir():
        selected.append(Path(source_directory))
    zip_root = Path(source_zip_directory) if source_zip_directory is not None else None
    if zip_root is not None and zip_root.is_dir():
        selected.extend(sorted(zip_root.glob('*.zip')))
    if not selected:
        raise FileNotFoundError('Set SOURCE_DIRECTORY or place source ZIP files in SOURCE_ZIP_DIRECTORY.')
    return selected
# SOURCE_SELECTION_ADAPTER_END
sources = select_image_sources(SOURCE_DIRECTORY, SOURCE_ZIP_DIRECTORY)
print(f'Runtime: {RUNTIME_ROOT}')
print(f'Sources: {len(sources)}; raw release: {RAW_RELEASE}; output: {PROCESSED_OUTPUT}')

## Prepare and verify the raw release

The Colab config selects all rows at 4X, 10X, and 40X and commits raw shards up to 512 MiB each. This can take a while because the source images must be read. A partial release is resumable, but must not be treated as complete cohort coverage.

In [ ]:
import json

CONFIG_PATH = RUNTIME_ROOT / 'configs' / 'data_shards_colab.json'
with CONFIG_PATH.open(encoding='utf-8') as stream:
    CONFIG = json.load(stream)

def run_cli(*arguments):
    command = [sys.executable, '-m', 'histology_data', *map(str, arguments)]
    subprocess.run(command, env=CLI_ENV, check=True)

prepare_args = ['prepare', '--metadata', METADATA, '--release', RAW_RELEASE, '--config', CONFIG_PATH]
for source in sources:
    prepare_args.extend(['--source', source])
run_cli(*prepare_args)
run_cli('verify', '--release', RAW_RELEASE)

## Process committed shards

Each shard is staged and processed under `/content/histology-work`; verified outputs and per-shard commit records are published to Drive. The default config processes all committed shards. Set `CONFIG['process']['max_shards']` to a positive integer before running the next cell to bound one invocation. Reruns verify existing commits and advance to new work.

In [ ]:
process_config_path = WORK_ROOT / 'data_shards_process.json'
with process_config_path.open('w', encoding='utf-8') as stream:
    json.dump(CONFIG, stream, ensure_ascii=False, indent=2)
process_args = [
    'process',
    '--release', RAW_RELEASE,
    '--work-root', WORK_ROOT,
    '--output', PROCESSED_OUTPUT,
    '--config', process_config_path,
]
if KEEP_WORK:
    process_args.append('--keep-work')
run_cli(*process_args)
print('Shard processing pass finished. Review the QC records before any downstream modeling work.')